# Res-UNet for Multi-Class Brain Tumor Segmentation

This notebook builds on the plain **U-Net** implemented in the previous session and upgrades it to a **Res-UNet** (Residual U-Net).

**What changes compared to plain U-Net?**
Only the *architecture* of the building block changes. The overall encoder–bridge–decoder shape, the dataset pipeline, the loss, and the training loop stay exactly the same as before. Instead of a plain `Conv -> BN -> ReLU -> Conv -> BN -> ReLU` block, every block becomes a **residual block**: the input of the block is added back to its output through a *shortcut (skip) connection*.

**Why bother with residual blocks?**
- They make it easier to train deeper networks (the gradient has a direct path back through the shortcut, so it doesn't have to only flow through the stacked convolutions).
- In practice, for segmentation tasks like this one, they tend to **lower the loss and increase accuracy/IoU** compared to the vanilla U-Net, for roughly the same amount of code.

The dataset used here is the same **Brain Tumor Segmentation Dataset** as before, with 4 classes (background + 3 tumor types), so the setup (loading, masks, class imbalance handling) is reused unchanged from the U-Net notebook.


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
from keras.models import Model
from keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D,concatenate, BatchNormalization, Add, Activation
from tensorflow.keras import backend as K
from sklearn.model_selection import train_test_split

# Load Dataset

In [ ]:
DATASET_PATH = './Brain Tumor Segmentation Dataset/'

In [ ]:
image_base_path = os.path.join(DATASET_PATH, 'image')
image_base_path

In [ ]:
mask_base_path = os.path.join(DATASET_PATH, 'mask')
mask_base_path

In [ ]:
class_dirs = [d for d in os.listdir(image_base_path) if os.path.isdir(os.path.join(image_base_path, d))]
class_dirs

In [ ]:
all_image_paths = []
all_mask_paths = []
all_labels = []

for i in range(len(class_dirs)):
    class_dir = class_dirs[i]

    image_folder = os.path.join(image_base_path, class_dir)
    mask_folder = os.path.join(mask_base_path, class_dir)

    label = int(class_dir)

    for file_name in os.listdir(image_folder):
        if file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
            img_path = os.path.join(image_folder, file_name)
            
            base_name, extension = os.path.splitext(file_name)
            mask_filename = f"{base_name}_m{extension}"
            mask_path = os.path.join(mask_folder, mask_filename)
            
            if os.path.exists(mask_path):
                all_image_paths.append(img_path)
                all_mask_paths.append(mask_path)
                all_labels.append(label)

In [ ]:
all_image_paths[0]

In [ ]:
all_mask_paths[0]

In [ ]:
all_labels[0]

In [ ]:
len(all_image_paths)

In [ ]:
train_images, val_images, train_masks, val_masks, train_labels, val_labels = train_test_split(
    all_image_paths, all_mask_paths, all_labels, test_size=0.1, random_state=42, stratify=all_labels
)

In [ ]:
len(train_images)

In [ ]:
len(val_images)

# Masks Pixel Check

In [ ]:
problematic_masks = []
all_unique_values = set()

for mask_path in all_mask_paths:
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    unique_values = np.unique(mask)
    
    all_unique_values.update(unique_values)
    
    extra_values = set(unique_values) - {0, 255}
    
    if extra_values:
        problematic_masks.append((mask_path, unique_values))


print(f"Global unique pixel values found in all masks: {sorted(list(all_unique_values))}")

In [ ]:
all_pixels_list = []

for mask_path in all_mask_paths:
    mask_image = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask_image is not None:
        all_pixels_list.append(mask_image.ravel())

all_pixel_values = np.concatenate(all_pixels_list)

plt.figure(figsize=(12, 7))
plt.hist(all_pixel_values, bins=256, range=[0, 256], color='blue')
plt.title('Histogram of Pixel Values for ALL Masks in the Dataset')
plt.xlabel('Pixel Value (0=Black, 255=White)')
plt.ylabel('Total Number of Pixels (Frequency)')
plt.grid(True, alpha=0.5)
plt.savefig("all_masks_histogram.png")

In [ ]:
IMG_SIZE = (256, 256)
BATCH_SIZE = 16

In [ ]:
def load_and_preprocess_multiclass(image_path, mask_path, label):
    IMG_SIZE = (256, 256)
    
    img = tf.io.read_file(image_path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, IMG_SIZE)

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    
    mask = tf.where(mask > 128, tf.cast(label, tf.uint8), tf.cast(0, tf.uint8))
    
    mask = tf.image.resize(mask, IMG_SIZE, method='nearest')
    
    return img, mask

In [ ]:
def augment_photometric(image, mask):
  
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    image = tf.clip_by_value(image, 0.0, 1.0)
    
    return image, mask

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_masks, train_labels))
train_dataset = train_dataset.map(load_and_preprocess_multiclass, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.map(augment_photometric, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(buffer_size=1000).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

# imbalance Data

In [ ]:
class_counts = {0: 0, 1: 0, 2: 0, 3: 0}
total_pixels = 0

for images_batch, masks_batch in train_dataset:
    for mask in masks_batch:
        mask_np = mask.numpy()
        total_pixels += mask_np.size
        unique, counts = np.unique(mask_np, return_counts=True)
        for u, c in zip(unique, counts):
            if u in class_counts:
                class_counts[u] += c

In [ ]:
total_pixels

In [ ]:
class_counts

## Median Frequency Balancing 

In [ ]:
class_frequencies = {cls: count / total_pixels for cls, count in class_counts.items() if count > 0}

frequencies = list(class_frequencies.values())
median_frequency = np.median(frequencies)

mfb_weights = {}
for cls, freq in class_frequencies.items():
    mfb_weights[cls] = median_frequency / freq

print(mfb_weights)

In [ ]:
val_dataset = tf.data.Dataset.from_tensor_slices((val_images, val_masks, val_labels))
val_dataset = val_dataset.map(load_and_preprocess_multiclass, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
print(f"Train dataset: {train_dataset}")
print(f"Validation dataset: {val_dataset}")

# Masks sample

In [ ]:
for images, masks in train_dataset.take(2): 
    
    sample_image = images[0]
    sample_mask = masks[0]
    print(np.unique(sample_mask))


    plt.figure(figsize=(8, 4))
    
    plt.subplot(1, 2, 1)
    plt.title("Sample Image")
    plt.imshow(sample_image, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title("Multi-class Mask")
    plt.imshow(np.squeeze(sample_mask), cmap='jet', vmin=0, vmax=3) 
    plt.axis('off')

    plt.show()

In [ ]:
sample_mask.shape

# U-NET

## From U-Net Blocks to Residual Blocks

Before writing the model, let's look at the **repeating pattern** used inside every residual block, since this is the only real difference from plain U-Net.

A "full" residual block repeats this pattern:

```
BatchNorm -> ReLU -> Conv -> BatchNorm -> ReLU -> Conv -> Add(shortcut)
```

Notice that the same sequence (`BN -> ReLU -> Conv`) appears **twice**, followed by adding the original input (the *shortcut*) to the result. Because this sequence is repeated everywhere in the network, it makes sense to wrap it in a reusable Python **function** instead of writing it out by hand for every stage of the encoder/decoder — that keeps the model definition short and consistent.

There is one exception: the **very first block** of the network (right after the input layer) does **not** have a `BatchNorm -> ReLU` before the first convolution, since there is no previous activation to normalize yet. That's why we define two functions below: `residual_block_first` for the first stage, and `residual_block` (the general, reusable one) for every stage after that.

### Handling the shortcut when shapes don't match
The shortcut path adds the block's input directly to its output, so both tensors must have **the exact same shape** (same number of channels, same spatial size). This is trivially true when the number of filters (channels) doesn't change from input to output. But whenever the block changes the number of channels (e.g. going from 32 filters to 64), a plain element-wise `Add` is no longer possible.

The fix is a **1x1 convolution** on the shortcut path: a 1x1 conv doesn't look at neighboring pixels, its only job is to project the number of channels up or down so the shortcut matches the main path's shape before the `Add`. This 1x1 projection is only applied *when needed* (i.e. when the input/output channel counts differ, or when a stride is used to downsample).


In [ ]:
def residual_block_first(x, filters):
    shortcut = x

    x = Conv2D(filters, (3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(filters, (3, 3), padding='same')(x)

    if K.int_shape(shortcut)[-1] != filters:
        shortcut = Conv2D(filters, (1, 1), padding='same')(shortcut)

    x = Add()([x, shortcut])
    return x

### General Residual Block (used everywhere except the first stage)

This is the reusable block described above:
1. `BatchNormalization -> ReLU -> Conv2D` (first pass)
2. `BatchNormalization -> ReLU -> Conv2D` (second pass)
3. If the number of channels changed (or a `stride` was used to downsample), project the shortcut with a `1x1 Conv2D` so its shape matches.
4. `Add` the (possibly projected) shortcut to the main path's output.

The `stride` argument lets this same function optionally be used for downsampling as an alternative to `MaxPooling2D`, although in the architecture below we still use pooling for downsampling and keep `stride=1` inside the blocks.


In [ ]:
def residual_block(x, filters, stride=1):
    shortcut = x

    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(filters, (3, 3), strides=stride, padding='same')(x)

    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(filters, (3, 3), padding='same')(x)

    if K.int_shape(shortcut)[-1] != filters or stride != 1:
        shortcut = Conv2D(filters, (1, 1), strides=stride, padding='same')(shortcut)

    x = Add()([x, shortcut])
    return x

## Building the Res-UNet Architecture

We now assemble the full network using the two block functions above. The overall shape is identical to a normal U-Net — an **encoder** that downsamples, a **bridge** at the bottleneck, and a **decoder** that upsamples and reuses encoder features via skip connections — but every convolutional stage is now a residual block.

**Encoder path** (each stage: residual block, then downsample):
- `residual_block_first` (32 filters) on the raw input, then `MaxPooling2D` — pooling is used for downsampling (rather than a strided convolution) to keep things simple and directly comparable to the plain U-Net.
- `residual_block` (64 filters) then pool.
- `residual_block` (128 filters) then pool.

**Bridge** (bottleneck, no pooling): `residual_block` with 256 filters.

**Decoder path** (each stage: upsample, concatenate with the matching encoder features, then a residual block):
- `UpSampling2D` doubles the spatial size using interpolation (a computationally cheap alternative to `Conv2DTranspose`, with no extra learnable parameters).
- The upsampled tensor is `concatenate`d with the encoder output **at the matching resolution** (this is the classic U-Net skip connection, unchanged here).
- The concatenated tensor is then passed through a `residual_block` to fuse the features.

This upsample -> concatenate -> residual block pattern repeats symmetrically to the encoder, mirroring the number of filters back down: 128 -> 64 -> 32.

**Output layer:** a final `1x1 Conv2D` with **4 filters** (one per class) followed by **softmax**, since this is a multi-class segmentation problem (4 classes). Note that the original Res-UNet paper uses `sigmoid`, because it was designed for **binary** segmentation. Here we use `softmax` instead, since our masks have 4 mutually-exclusive classes rather than a single binary foreground/background mask.

The resulting model has about **2.2 million parameters** (roughly `2,216,xxx`), which you can confirm from the `model.summary()` output below.


In [ ]:
inputs = Input(shape=(256, 256, 1))

# Encoder
res1 = residual_block_first(inputs, filters=32)
pool1 = MaxPooling2D((2, 2))(res1)

res2 = residual_block(pool1, filters=64)
pool2 = MaxPooling2D((2, 2))(res2)

res3 = residual_block(pool2, filters=128)
pool3 = MaxPooling2D((2, 2))(res3)

# Bridge
bridge = residual_block(pool3, filters=256)

# Decoder
up1 = UpSampling2D((2, 2))(bridge)
concat1 = concatenate([up1, res3])
res4 = residual_block(concat1, filters=128)

up2 = UpSampling2D((2, 2))(res4)
concat2 = concatenate([up2, res2])
res5 = residual_block(concat2, filters=64)

up3 = UpSampling2D((2, 2))(res5)
concat3 = concatenate([up3, res1])
res6 = residual_block(concat3, filters=32)

# Output
outputs = Conv2D(4, (1, 1), activation='softmax')(res6)

model = Model(inputs=[inputs], outputs=[outputs])
model.summary()

In [ ]:
sparse_mean_iou = tf.keras.metrics.MeanIoU(num_classes=4,sparse_y_pred=False)

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy', sparse_mean_iou])

In [ ]:
from tensorflow.keras import callbacks
callbacks = [
    callbacks.ModelCheckpoint('./bestmodel_ResUNet.keras',
                             monitor="val_loss",
                             verbose=0,            
                             save_best_only=True),
]

In [ ]:
history = model.fit(
    train_dataset,
    epochs=60,
    validation_data=val_dataset,
    class_weight=mfb_weights,
    callbacks=callbacks
)

In [ ]:
history_dict = history.history

In [ ]:
plt.plot(history_dict['loss'], label='Training Loss')
plt.plot(history_dict['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.plot(history.history['mean_io_u'], label='Training Mean IoU')
plt.plot(history.history['val_mean_io_u'], label='Validation Mean IoU')
plt.xlabel('Epochs')
plt.ylabel('Mean IoU')
plt.title('Training and Validation Mean IoU')
plt.legend()
plt.grid(True)
plt.show()

# Evaluation

In [ ]:
from tensorflow.keras.models import load_model

best_model = load_model('./bestmodel.keras',)

In [ ]:
sample_index = 200
sample_image_path = train_images[sample_index]
sample_mask_path = train_masks[sample_index]
sample_label = train_labels[sample_index]

img = tf.io.read_file(sample_image_path)
img = tf.image.decode_png(img, channels=1)
img = tf.image.convert_image_dtype(img, tf.float32)

IMG_SIZE = (256, 256)
img_resized = tf.image.resize(img, IMG_SIZE)

img_for_prediction = tf.expand_dims(img_resized, axis=0)


prediction = best_model.predict(img_for_prediction)
confidence_threshold = 0.99

max_probs = np.max(prediction, axis=-1)
predicted_labels = np.argmax(prediction, axis=-1)

predicted_mask = np.where(max_probs < confidence_threshold, 0, predicted_labels)

predicted_mask = np.squeeze(predicted_mask)


_, true_mask = load_and_preprocess_multiclass(sample_image_path, sample_mask_path, sample_label)
true_mask = np.squeeze(true_mask.numpy())


plt.figure(figsize=(15, 5))


plt.subplot(1, 3, 1)
plt.title("Original Image")
plt.imshow(img_resized,cmap='gray')
plt.axis('off')


plt.subplot(1, 3, 2)
plt.title("True Mask")
plt.imshow(true_mask, cmap='jet', vmin=0, vmax=3)
plt.axis('off')


plt.subplot(1, 3, 3)
plt.title(f"Predicted Mask (Threshold={confidence_threshold})")
plt.imshow(predicted_mask, cmap='jet', vmin=0, vmax=3)
plt.axis('off')

plt.tight_layout()
plt.savefig("prediction_sample_with_threshold.png")
plt.show()

print(f"Unique values in True Mask: {np.unique(true_mask)}")
print(f"Unique values in Predicted Mask (with threshold): {np.unique(predicted_mask)}")